In [1]:
!pip install opencv-python-headless mtcnn scikit-learn xgboost tensorflow

  Using cached opencv_python_headless-4.11.0.86-cp37-abi3-macosx_13_0_arm64.whl.metadata (20 kB)
Using cached opencv_python_headless-4.11.0.86-cp37-abi3-macosx_13_0_arm64.whl (37.3 MB)


In [3]:
import os
import cv2
import numpy as np
from mtcnn import MTCNN
from collections import defaultdict
from tensorflow.keras.applications.xception import Xception, preprocess_input as xception_preprocess
from tensorflow.keras.applications.efficientnet import EfficientNetB7, preprocess_input as efficientnet_preprocess
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Input video paths
real_video_path = '/Users/vankayalmeghashree/Downloads/deepfakedataset_2/DFD_original sequences'
fake_video_path = '/Users/vankayalmeghashree/Downloads/deepfakedataset_2/DFD_manipulated_sequences/DFD_manipulated_sequences'

# Output face folders — corrected
extracted_faces_real = '/Users/vankayalmeghashree/Downloads/deepfakedataset_2/extracted_faces_real'
extracted_faces_fake = '/Users/vankayalmeghashree/Downloads/deepfakedataset_2/extracted_faces_fake'

# Create output folders if they don't exist
os.makedirs(extracted_faces_real, exist_ok=True)
os.makedirs(extracted_faces_fake, exist_ok=True)

print("✅ Libraries imported and paths configured.")


✅ Libraries imported and paths configured.


In [6]:
from concurrent.futures import ThreadPoolExecutor
import multiprocessing

detector = MTCNN()

def extract_faces_from_video(video_path, output_folder, max_faces=60):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved_faces = 0

    while cap.isOpened() and saved_faces < max_faces:
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        faces = detector.detect_faces(rgb_frame)

        for i, face in enumerate(faces):
            if saved_faces >= max_faces:
                break
            x, y, w, h = face['box']
            x, y = max(0, x), max(0, y)
            face_img = rgb_frame[y:y+h, x:x+w]
            face_img = cv2.resize(face_img, (224, 224))
            filename = f"{os.path.basename(video_path).replace('.mp4','')}_{frame_count}_{i}.jpg"
            cv2.imwrite(os.path.join(output_folder, filename), cv2.cvtColor(face_img, cv2.COLOR_RGB2BGR))
            saved_faces += 1

        frame_count += 1
    cap.release()

# Wrap the args for each task
def process_single_video(args):
    video_path, output_folder = args
    print(f"🎥 Extracting faces from: {os.path.basename(video_path)}")
    extract_faces_from_video(video_path, output_folder)

# Parallel processing for one folder
def process_video_folder_parallel(video_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    video_files = [os.path.join(video_folder, f) for f in os.listdir(video_folder) if f.endswith('.mp4')]
    args = [(video_path, output_folder) for video_path in video_files]

    max_workers = multiprocessing.cpu_count()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        executor.map(process_single_video, args)

# Run for both real and fake
process_video_folder_parallel(real_video_path, extracted_faces_real)
process_video_folder_parallel(fake_video_path, extracted_faces_fake)

print("✅ Done extracting faces from all videos using parallel threads.")

🎥 Extracting faces from: 07__exit_phone_room.mp4🎥 Extracting faces from: 09__kitchen_pan.mp4

🎥 Extracting faces from: 02__walking_down_street_outside_angry.mp4
🎥 Extracting faces from: 12__talking_angry_couch.mp4
🎥 Extracting faces from: 11__podium_speech_happy.mp4
🎥 Extracting faces from: 26__podium_speech_happy.mp4
🎥 Extracting faces from: 24__kitchen_still.mp4
🎥 Extracting faces from: 26__outside_talking_pan_laughing.mp4
🎥 Extracting faces from: 11__walking_down_street_outside_angry.mp4
🎥 Extracting faces from: 16__walk_down_hall_angry.mp4
🎥 Extracting faces from: 17__talking_against_wall.mp4
🎥 Extracting faces from: 18__secret_conversation.mp4
🎥 Extracting faces from: 14__walking_and_outside_surprised.mp4
🎥 Extracting faces from: 05__outside_talking_still_laughing.mp4
🎥 Extracting faces from: 21__talking_angry_couch.mp4
🎥 Extracting faces from: 12__outside_talking_pan_laughing.mp4
🎥 Extracting faces from: 24__walking_down_street_outside_angry.mp4
🎥 Extracting faces from: 21__outsi

In [7]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load image paths
real_face_paths = [os.path.join(extracted_faces_real, f) for f in os.listdir(extracted_faces_real) if f.endswith('.jpg')]
fake_face_paths = [os.path.join(extracted_faces_fake, f) for f in os.listdir(extracted_faces_fake) if f.endswith('.jpg')]

# Combine and label
all_face_paths = real_face_paths + fake_face_paths
labels = [0] * len(real_face_paths) + [1] * len(fake_face_paths)

# Load and preprocess images
def read_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))  # Ensure size
    return img

print("📥 Reading images into memory...")
all_images = [read_image(p) for p in all_face_paths]
all_images = np.array(all_images)

# Apply augmentation
print("✨ Applying augmentation...")
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest'
)

augmented_images = []
for img in all_images:
    img = np.expand_dims(img, axis=0)
    aug_img = next(datagen.flow(img, batch_size=1))[0].astype(np.uint8)
    augmented_images.append(aug_img)

augmented_images = np.array(augmented_images)

print(f"✅ Loaded {len(augmented_images)} augmented images.")


📥 Reading images into memory...
✨ Applying augmentation...
✅ Loaded 39360 augmented images.


In [9]:
from tensorflow.keras.applications.xception import Xception, preprocess_input as xception_preprocess
from tensorflow.keras.applications.efficientnet import EfficientNetB7, preprocess_input as efficientnet_preprocess
from tensorflow.keras.models import Model
import numpy as np

# Load pretrained models without top layers
print("📥 Loading pretrained CNN models...")
xception_base = Xception(weights='imagenet', include_top=False, pooling='avg')
xception_model = Model(inputs=xception_base.input, outputs=xception_base.output)

efficientnet_base = EfficientNetB7(weights='imagenet', include_top=False, pooling='avg')
efficientnet_model = Model(inputs=efficientnet_base.input, outputs=efficientnet_base.output)

# Sequential extraction function
def extract_features_batch(images, model, preprocess_func):
    print(f"🔍 Extracting features using {model.name} (sequential)...")
    features = []
    for img in images:
        img_proc = preprocess_func(img)
        img_proc = np.expand_dims(img_proc, axis=0)
        feat = model.predict(img_proc, verbose=0)[0]
        features.append(feat)
    return np.array(features)

# Run extraction
xception_features = extract_features_batch(augmented_images, xception_model, xception_preprocess)
efficientnet_features = extract_features_batch(augmented_images, efficientnet_model, efficientnet_preprocess)

# Combine the features
stacked_features = np.hstack((xception_features, efficientnet_features))
print(f"✅ Feature extraction complete. Final shape: {stacked_features.shape}")


📥 Loading pretrained CNN models...
🔍 Extracting features using functional_2 (sequential)...
🔍 Extracting features using functional_3 (sequential)...
✅ Feature extraction complete. Final shape: (39360, 4608)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import numpy as np

print("🔁 Starting hybrid model training...")

# 1. Split the data
print("🔀 Splitting data into Train / Val / Test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    stacked_features, labels, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# ✅ Convert labels to NumPy arrays
y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

print(f"🧪 Dataset shapes — Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# 2. Train MLP (Neural Network)
print("🔧 Training MLP...")
mlp = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(2, activation='softmax')
])
mlp.compile(optimizer=Adam(learning_rate=0.0001),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

mlp.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_val, y_val), verbose=0)
print("✅ MLP training complete.")

mlp_val_preds = mlp.predict(X_val)
mlp_val_pred_classes = np.argmax(mlp_val_preds, axis=1)
print("📈 MLP validation predictions generated.")

# 3. Train XGBoost
print("🧠 Training XGBoost Classifier...")
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_val_pred = xgb.predict(X_val)
print("✅ XGBoost training complete.")

# 4. Train Random Forest
print("🌲 Training Random Forest Classifier...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_val_pred = rf.predict(X_val)
print("✅ Random Forest training complete.")

# 5. Stack predictions from all 3 on validation set
print("📊 Stacking predictions from MLP, XGBoost, and RF...")
stacked_val_preds = np.vstack([
    mlp_val_pred_classes,
    xgb_val_pred,
    rf_val_pred
]).T  # shape: (samples, 3)

# 6. Train meta-learner (Logistic Regression)
print("🤖 Training Logistic Regression as Meta-Learner...")
meta_clf = LogisticRegression()
meta_clf.fit(stacked_val_preds, y_val)
print("✅ Meta-learner training complete and ready for final test evaluation.")


🔁 Starting hybrid model training...
🔀 Splitting data into Train / Val / Test sets...
🧪 Dataset shapes — Train: (25190, 4608), Val: (6298, 4608), Test: (7872, 4608)
🔧 Training MLP...
✅ MLP training complete.
197/197 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
📈 MLP validation predictions generated.
🧠 Training XGBoost Classifier...


/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [00:56:16] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ XGBoost training complete.
🌲 Training Random Forest Classifier...
✅ Random Forest training complete.
📊 Stacking predictions from MLP, XGBoost, and RF...
🤖 Training Logistic Regression as Meta-Learner...
✅ Meta-learner training complete and ready for final test evaluation.


In [12]:
print("🧪 Starting final test evaluation...")

# Step 1: Get predictions on test set
print("🔍 Predicting with base models on test set...")

mlp_test_preds = mlp.predict(X_test)
mlp_test_classes = np.argmax(mlp_test_preds, axis=1)

xgb_test_classes = xgb.predict(X_test)
rf_test_classes = rf.predict(X_test)

# Step 2: Stack base predictions
print("📊 Stacking base predictions for final classification...")
stacked_test_preds = np.vstack([
    mlp_test_classes,
    xgb_test_classes,
    rf_test_classes
]).T  # shape: (samples, 3)

# Step 3: Meta-learner predicts final class
print("🤖 Meta-learner making final predictions...")
final_preds = meta_clf.predict(stacked_test_preds)

# Step 4: Metrics
print("📈 Classification Report:\n")
print(classification_report(y_test, final_preds, target_names=["Real", "Fake"]))

acc = accuracy_score(y_test, final_preds)
print(f"✅ Final Test Accuracy: {acc:.4f}")


🧪 Starting final test evaluation...
🔍 Predicting with base models on test set...
246/246 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
📊 Stacking base predictions for final classification...
🤖 Meta-learner making final predictions...
📈 Classification Report:

              precision    recall  f1-score   support

        Real       0.93      0.97      0.95      4359
        Fake       0.96      0.91      0.93      3513

    accuracy                           0.94      7872
   macro avg       0.94      0.94      0.94      7872
weighted avg       0.94      0.94      0.94      7872

✅ Final Test Accuracy: 0.9403


In [26]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

print("🧪 Starting final test evaluation...")

# Step 1: Get predictions on test set
print("🔍 Predicting with base models on test set...")
mlp_test_preds = mlp.predict(X_test, verbose=0)
mlp_test_classes = np.argmax(mlp_test_preds, axis=1)
xgb_test_proba = xgb.predict_proba(X_test)
xgb_test_classes = np.argmax(xgb_test_proba, axis=1)
rf_test_proba = rf.predict_proba(X_test)
rf_test_classes = np.argmax(rf_test_proba, axis=1)

# Step 2: Stack base predictions
print("📊 Stacking base predictions for final classification...")
stacked_test_preds = np.vstack([
    mlp_test_classes,
    xgb_test_classes,
    rf_test_classes
]).T

# Step 3: Meta-learner final prediction
print("🤖 Meta-learner making final predictions...")
final_preds = meta_clf.predict(stacked_test_preds)

# Step 4: Classification Report
print("\n📈 Classification Report:\n")
print(classification_report(y_test, final_preds, target_names=["Real", "Fake"]))
acc = accuracy_score(y_test, final_preds)
print(f"✅ Final Test Accuracy: {acc:.4f}")

# Step 5: Confusion Matrix
print("📊 Generating confusion matrix...")
cm = confusion_matrix(y_test, final_preds)
labels = ["Real", "Fake"]

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/confusion_matrix.png")
plt.close()

# Step 6: ROC Curve
print("📉 Generating ROC curve...")
from sklearn.metrics import roc_auc_score

mlp_auc = roc_auc_score(y_test, mlp_test_preds[:, 1])
xgb_auc = roc_auc_score(y_test, xgb_test_proba[:, 1])
rf_auc = roc_auc_score(y_test, rf_test_proba[:, 1])

fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp_test_preds[:, 1])
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_test_proba[:, 1])
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_test_proba[:, 1])

plt.figure(figsize=(7, 5))
plt.plot(fpr_mlp, tpr_mlp, label=f"MLP (AUC={mlp_auc:.2f})")
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={xgb_auc:.2f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={rf_auc:.2f})")
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/roc_curve.png")
plt.close()

# Step 7: Precision-Recall Curve
print("📐 Generating Precision-Recall curve...")
prec_mlp, rec_mlp, _ = precision_recall_curve(y_test, mlp_test_preds[:, 1])
prec_xgb, rec_xgb, _ = precision_recall_curve(y_test, xgb_test_proba[:, 1])
prec_rf, rec_rf, _ = precision_recall_curve(y_test, rf_test_proba[:, 1])

plt.figure(figsize=(7, 5))
plt.plot(rec_mlp, prec_mlp, label="MLP")
plt.plot(rec_xgb, prec_xgb, label="XGBoost")
plt.plot(rec_rf, prec_rf, label="Random Forest")
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/precision_recall_curve.png")
plt.close()

# Step 8: Accuracy Comparison
print("📊 Generating accuracy comparison chart...")
mlp_acc = accuracy_score(y_test, mlp_test_classes)
xgb_acc = accuracy_score(y_test, xgb_test_classes)
rf_acc = accuracy_score(y_test, rf_test_classes)
meta_acc = accuracy_score(y_test, final_preds)

model_names = ["MLP", "XGBoost", "Random Forest", "Meta Learner"]
accs = [mlp_acc, xgb_acc, rf_acc, meta_acc]

plt.figure(figsize=(6, 4))
sns.barplot(x=accs, y=model_names, palette="viridis")
plt.title("Accuracy Comparison")
plt.xlabel("Accuracy")
plt.tight_layout()
plt.savefig("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/accuracy_comparison.png")
plt.close()

print("✅ All visual reports saved to the output folder.")


🧪 Starting final test evaluation...
🔍 Predicting with base models on test set...
📊 Stacking base predictions for final classification...
🤖 Meta-learner making final predictions...

📈 Classification Report:

              precision    recall  f1-score   support

        Real       0.93      0.97      0.95      4359
        Fake       0.96      0.91      0.93      3513

    accuracy                           0.94      7872
   macro avg       0.94      0.94      0.94      7872
weighted avg       0.94      0.94      0.94      7872

✅ Final Test Accuracy: 0.9403
📊 Generating confusion matrix...
📉 Generating ROC curve...
📐 Generating Precision-Recall curve...
📊 Generating accuracy comparison chart...
✅ All visual reports saved to the output folder.


/var/folders/yx/lc7x41b56dl7mlvt4mm1ppvw0000gn/T/ipykernel_8633/1366715104.py:107: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=accs, y=model_names, palette="viridis")


In [13]:
import os
os.makedirs("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models", exist_ok=True)

In [14]:
import joblib

# 1. Save MLP (Keras model)
mlp.save("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/mlp_model.h5")
print("💾 Saved MLP model (Keras)")

# 2. Save XGBoost
joblib.dump(xgb, "/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/xgb_model.pkl")
print("💾 Saved XGBoost model")

# 3. Save Random Forest
joblib.dump(rf, "/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/rf_model.pkl")
print("💾 Saved Random Forest model")

# 4. Save Logistic Regression (Meta-Learner)
joblib.dump(meta_clf, "/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/meta_learner.pkl")
print("💾 Saved Logistic Regression (meta-learner)")

💾 Saved MLP model (Keras)
💾 Saved XGBoost model
💾 Saved Random Forest model
💾 Saved Logistic Regression (meta-learner)


In [16]:
from tensorflow.keras.models import load_model
import joblib

mlp = load_model("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/mlp_model.h5")
xgb = joblib.load("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/xgb_model.pkl")
rf = joblib.load("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/rf_model.pkl")
meta_clf = joblib.load("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/meta_learner.pkl")


In [17]:
from tensorflow.keras.optimizers import Adam

mlp.compile(optimizer=Adam(learning_rate=0.0001),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

In [18]:
#Testing

In [23]:
import cv2
import numpy as np
from mtcnn import MTCNN
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.models import Model
from tensorflow.keras.applications import Xception, EfficientNetB7
import joblib
from collections import Counter
import os
from tensorflow.keras.optimizers import Adam

# Load models
mlp = load_model("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/mlp_model.h5")
xgb = joblib.load("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/xgb_model.pkl")
rf = joblib.load("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/rf_model.pkl")
meta_clf = joblib.load("/Users/vankayalmeghashree/Downloads/deepfakedataset_2/models/meta_learner.pkl")

mlp.compile(optimizer=Adam(learning_rate=0.0001),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])


# Load feature extractors
#xception_model = Model(inputs=Xception(weights='imagenet', include_top=False, pooling='avg').input,
                       #outputs=Xception(weights='imagenet', include_top=False, pooling='avg').output)

#efficientnet_model = Model(inputs=EfficientNetB7(weights='imagenet', include_top=False, pooling='avg').input,
                           #outputs=EfficientNetB7(weights='imagenet', include_top=False, pooling='avg').output)
xception_model = Xception(weights='imagenet', include_top=False, pooling='avg')
efficientnet_model = EfficientNetB7(weights='imagenet', include_top=False, pooling='avg')

detector = MTCNN()

def extract_faces_from_video(video_path, max_faces=60):
    faces = []
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved_faces = 0

    while cap.isOpened() and saved_faces < max_faces:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        detections = detector.detect_faces(rgb)

        for face in detections:
            if saved_faces >= max_faces:
                break
            x, y, w, h = face['box']
            x, y = max(0, x), max(0, y)
            face_crop = rgb[y:y+h, x:x+w]
            face_crop = cv2.resize(face_crop, (224, 224))
            faces.append(face_crop)
            saved_faces += 1

        frame_count += 1
    cap.release()
    return faces

def extract_features(faces):
    xception_feats = []
    efficientnet_feats = []

    for img in faces:
        x_input = xception_preprocess(np.expand_dims(img.astype(np.float32), axis=0))
        e_input = efficientnet_preprocess(np.expand_dims(img.astype(np.float32), axis=0))
        x_feat = xception_model.predict(x_input, verbose=0)[0]
        e_feat = efficientnet_model.predict(e_input, verbose=0)[0]
        stacked = np.concatenate([x_feat, e_feat])
        xception_feats.append(stacked)

    return np.array(xception_feats)

def predict_video_fake_or_real(video_path):
    print(f"🎥 Processing video: {video_path}")
    faces = extract_faces_from_video(video_path)
    if not faces:
        return "❌ No faces detected in video."

    print(f"🧠 Extracting features from {len(faces)} faces...")
    features = extract_features(faces)

    print(f"🔮 Making predictions with base models...")
    mlp_preds = mlp.predict(features, verbose=0)
    mlp_classes = np.argmax(mlp_preds, axis=1)

    xgb_preds = xgb.predict(features)
    rf_preds = rf.predict(features)

    stacked_preds = np.vstack([mlp_classes, xgb_preds, rf_preds]).T
    final_preds = meta_clf.predict(stacked_preds)

    # Majority vote
    count = Counter(final_preds)
    label = count.most_common(1)[0][0]
    result = "Fake" if label == 1 else "Real"
    print(f"✅ Final Decision: {result} ({count[0]} Real, {count[1]} Fake)")

    return result